# Step 1
Download reference genome

In [4]:
!rsync -a -P rsync://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr10.fa.gz ./

receiving incremental file list
chr10.fa.gz
     43,157,332 100%   50.01MB/s    0:00:00 (xfr#1, to-chk=0/1)


Decompress

In [ ]:
!gunzip chr10.fa.gz

Download samples

In [103]:
!curl -L -o illumina.fq.bz2 https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/illumina.fq.bz2
!curl -L -o pacbio.fq.bz2 https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/pacbio.fq.bz2

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 20.7M  100 20.7M    0     0  16.3M      0  0:00:01  0:00:01 --:--:-- 16.3M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 3253k  100 3253k    0     0  4180k      0 --:--:-- --:--:-- --:--:-- 4180k


Decompress

In [104]:
!bzip2 -dkf illumina.fq.bz2
!bzip2 -dkf pacbio.fq.bz2

# Step 2
Align samples to reference genome

Install minimap2

In [105]:
!curl -L  https://github.com/lh3/minimap2/releases/download/v2.30/minimap2-2.30_x64-linux.tar.bz2 -o minimap2.tar.bz2
!tar -jxvf minimap2.tar.bz2

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 2189k  100 2189k    0     0  5100k      0 --:--:-- --:--:-- --:--:-- 5100k
minimap2-2.30_x64-linux/
minimap2-2.30_x64-linux/cookbook.md
minimap2-2.30_x64-linux/paftools.js
minimap2-2.30_x64-linux/NEWS.md
minimap2-2.30_x64-linux/README.md
minimap2-2.30_x64-linux/k8
minimap2-2.30_x64-linux/FAQ.md
minimap2-2.30_x64-linux/minimap2.1
minimap2-2.30_x64-linux/LICENSE.txt
minimap2-2.30_x64-linux/minimap2
minimap2-2.30_x64-linux/README-js.md


Align with minimap2

In [111]:
!minimap2-2.30_x64-linux/minimap2 -ax map-ont chr10.fa pacbio.fq | samtools view -bS -o sample_pacbio.bam
!samtools sort -o sample_pacbio.bam sample_pacbio.bam
!samtools index sample_pacbio.bam

[M::mm_idx_gen::2.337*0.96] collected minimizers
[M::mm_idx_gen::2.769*1.26] sorted minimizers
[M::main::2.770*1.26] loaded/built the index for 1 target sequence(s)
[M::mm_mapopt_update::2.992*1.24] mid_occ = 178
[M::mm_idx_stat] kmer size: 15; skip: 10; is_hpc: 0; #seq: 1
[M::mm_idx_stat::3.135*1.22] distinct minimizers: 16061920 (79.91% are singletons); average occurrences: 1.563; average spacing: 5.329; total length: 133797422
[M::worker_pipeline::4.559*1.66] mapped 3063 sequences
[M::main] Version: 2.30-r1287
[M::main] CMD: minimap2-2.30_x64-linux/minimap2 -ax map-ont chr10.fa pacbio.fq
[M::main] Real time: 4.577 sec; CPU: 7.593 sec; Peak RSS: 1.155 GB


In [112]:
!minimap2-2.30_x64-linux/minimap2 -ax sr chr10.fa illumina.fq | samtools view -bS -o sample_illumina.bam
!samtools sort -o sample_illumina.bam sample_illumina.bam
!samtools index sample_illumina.bam

[M::mm_idx_gen::2.156*0.96] collected minimizers
[M::mm_idx_gen::2.437*1.17] sorted minimizers
[M::main::2.437*1.17] loaded/built the index for 1 target sequence(s)
[M::mm_mapopt_update::2.437*1.17] mid_occ = 1000
[M::mm_idx_stat] kmer size: 21; skip: 11; is_hpc: 0; #seq: 1
[M::mm_idx_stat::2.564*1.16] distinct minimizers: 19023001 (96.65% are singletons); average occurrences: 1.171; average spacing: 6.004; total length: 133797422
[M::worker_pipeline::10.125*1.99] mapped 309505 sequences
[M::main] Version: 2.30-r1287
[M::main] CMD: minimap2-2.30_x64-linux/minimap2 -ax sr chr10.fa illumina.fq
[M::main] Real time: 10.159 sec; CPU: 20.211 sec; Peak RSS: 1.241 GB


# Step 3
Call variants

Install freebayes

In [113]:
!curl -L https://github.com/freebayes/freebayes/releases/download/v1.3.10/freebayes-1.3.10-linux-amd64-static.gz -o freebayes.gz
!gunzip --force freebayes.gz
!chmod +x freebayes

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 2436k  100 2436k    0     0  5400k      0 --:--:-- --:--:-- --:--:-- 5400k


Use freebayes

In [114]:
!./freebayes -f chr10.fa sample_pacbio.bam > variants_pacbio.vcf
!./freebayes -f chr10.fa sample_illumina.bam > variants_illumina.vcf

# Step 4
Phase the variant VCF's

Clone and build HapCUT2

In [135]:
!rm -rf HapCUT2/
!git clone https://github.com/vibansal/HapCUT2.git

Cloning into 'HapCUT2'...
remote: Enumerating objects: 3167, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 3167 (delta 53), reused 69 (delta 29), pack-reused 3052 (from 1)
Receiving objects: 100% (3167/3167), 3.66 MiB | 8.46 MiB/s, done.
Resolving deltas: 100% (2118/2118), done.


In [151]:
!cd HapCUT2 && rm -rf htslib/
!cd HapCUT2 && git clone https://github.com/samtools/htslib.git
!cd HapCUT2/htslib && git submodule update --init --recursive

Cloning into 'htslib'...
remote: Enumerating objects: 18035, done.
remote: Counting objects: 100% (363/363), done.
remote: Compressing objects: 100% (196/196), done.
remote: Total 18035 (delta 252), reused 170 (delta 167), pack-reused 17672 (from 4)
Receiving objects: 100% (18035/18035), 13.78 MiB | 23.67 MiB/s, done.
Resolving deltas: 100% (12976/12976), done.
Submodule 'htscodecs' (https://github.com/samtools/htscodecs.git) registered for path 'htscodecs'
Cloning into '/home/liam/Bioinformatics/fall25-csc-bioinf/week5/HapCUT2/htslib/htscodecs'...
Submodule path 'htscodecs': checked out 'a815cd02bf0bada68800f32bfb0997d40f579026'


In [152]:
!cd HapCUT2/htslib && autoreconf -i && ./configure && make clean && make && make install

checking for gcc... gcc
checking whether the C compiler works... yes
checking for C compiler default output file name... a.out
checking for suffix of executables... 
checking whether we are cross compiling... no
checking for suffix of object files... o
checking whether the compiler supports GNU C... yes
checking whether gcc accepts -g... yes
checking for gcc option to enable C11 features... none needed
checking for ranlib... ranlib
checking for grep that handles long lines and -e... /usr/bin/grep
checking for C compiler warning flags... -Wall
checking for gcc options needed to detect all undeclared functions... none needed
checking for stdio.h... yes
checking for stdlib.h... yes
checking for string.h... yes
checking for inttypes.h... yes
checking for stdint.h... yes
checking for strings.h... yes
checking for sys/stat.h... yes
checking for sys/types.h... yes
checking for unistd.h... yes
checking for sys/param.h... yes
checking whether _XOPEN_SOURCE is declared... no
checking whether __g

In [153]:
!cd HapCUT2 && make HTSLIB=./htslib

cc -Wall -g -O3 -Wall -D_GNU_SOURCE -L./htslib -I./htslib -g build/bamread.o build/hapfragments.o build/hashtable.o build/readfasta.o build/readvariant.o -o build/extractHAIRS hairs-src/extracthairs.c -pthread -lhts -lm -lz -lcurl -llzma -lbz2
cc -c -Wall -g -O3 -Wall -D_GNU_SOURCE hapcut2-src/variantgraph.c -o build/variantgraph.o
cc -c -Wall -g -O3 -Wall -D_GNU_SOURCE hapcut2-src/readinputfiles.c -o build/readinputfiles.o
cc -c -Wall -g -O3 -Wall -D_GNU_SOURCE hapcut2-src/hapcontig.c -o build/hapcontig.o
cc -c -Wall -g -O3 -Wall -D_GNU_SOURCE hapcut2-src/fragments.c -o build/fragments.o
cc -c -Wall -g -O3 -Wall -D_GNU_SOURCE hapcut2-src/readvcf.c -o build/readvcf.o
cc -c -Wall -g -O3 -Wall -D_GNU_SOURCE hapcut2-src/pointerheap.c -o build/pointerheap.o
cc -c -Wall -g -O3 -Wall -D_GNU_SOURCE hapcut2-src/common.c -o build/common.o
cc -c -Wall -g -O3 -Wall -D_GNU_SOURCE hapcut2-src/hic.c -o build/hic.o
cc -Wall -g -O3 -Wall -D_GNU_SOURCE -L./htslib build/common.o build/hic.o build/varian

Now use HapCUT2

In [154]:
!./HapCUT2/build/extractHAIRS --bam sample_pacbio.bam --VCF variants_pacbio.vcf --out pacbio_fragments
!./HapCUT2/build/extractHAIRS --bam sample_illumina.bam --VCF variants_illumina.vcf --out illumina_fragments


Extracting haplotype informative reads from bamfiles sample_pacbio.bam minQV 13 minMQ 20 maxIS 1000 

VCF file variants_pacbio.vcf has 634 variants 
adding chrom chr10 to index 
vcffile variants_pacbio.vcf chromosomes 1 hetvariants 325 variants 634 
detected 15 variants with two non-reference alleles, these variants will not be phased
reading sorted bam/cram file sample_pacbio.bam 
processing reads mapped to chrom "chr10" 

Extracting haplotype informative reads from bamfiles sample_illumina.bam minQV 13 minMQ 20 maxIS 1000 

VCF file variants_illumina.vcf has 6017 variants 
adding chrom chr10 to index 
vcffile variants_illumina.vcf chromosomes 1 hetvariants 1256 variants 6017 
detected 49 variants with two non-reference alleles, these variants will not be phased
reading sorted bam/cram file sample_illumina.bam 
processing reads mapped to chrom "chr10" 
final cleanup of fragment list: 37939 current chrom 0 prev 0 


In [157]:
!./HapCUT2/build/HAPCUT2 --fragments pacbio_fragments --VCF variants_pacbio.vcf --output pacbio_phased.vcf
!./HapCUT2/build/HAPCUT2 --fragments illumina_fragments --VCF variants_illumina.vcf --output illumina_phased.vcf



[2025:11:04 00:27:37] input fragment file: pacbio_fragments
[2025:11:04 00:27:37] input variantfile (VCF format):variants_pacbio.vcf
[2025:11:04 00:27:37] haplotypes will be output to file: pacbio_phased.vcf
[2025:11:04 00:27:37] solution convergence cutoff: 5
[2025:11:04 00:27:37] read 634 variants from variants_pacbio.vcf file 
[2025:11:04 00:27:37] read fragment file and variant file: fragments 2349 variants 634
mean number of variants per read is 3.76 
[2025:11:04 00:27:37] building read-variant graph for phasing
Number of non-trivial connected components 14 max-Degree 235 connected variants 237 coverage-per-variant 37.308017 
[2025:11:04 00:27:37] fragments 2349 snps 634 component(blocks) 14
[2025:11:04 00:27:37] starting Max-Likelihood-Cut based haplotype assembly algorithm
[2025:11:04 00:27:37] starting to post-process phased haplotypes to further improve accuracy
[2025:11:04 00:27:37] starting to output phased haplotypes
[2025:11:04 00:27:37] OUTPUTTING PRUNED HAPLOTYPE ASSEM

Download HapCutToVCF

In [170]:
!wget https://github.com/fulcrumgenomics/fgbio/releases/download/3.0.0/fgbio-3.0.0.jar

--2025-11-05 15:37:29--  https://github.com/fulcrumgenomics/fgbio/releases/download/3.0.0/fgbio-3.0.0.jar
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/53011104/914df6da-f8e0-43b4-a22e-71205b6b96f6?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-11-06T00%3A34%3A30Z&rscd=attachment%3B+filename%3Dfgbio-3.0.0.jar&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-11-05T23%3A34%3A13Z&ske=2025-11-06T00%3A34%3A30Z&sks=b&skv=2018-11-09&sig=ysvPLrEP4rM8UV7rEzKOYT4MaE1jtIDW87Y3nJB6MUU%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc2MjM4NzY0OSwibmJmIjoxNzYyMzg1ODQ5LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG

In [178]:
!java -jar fgbio-3.0.0.jar HapCutToVcf \
  --input=illumina_phased.vcf \
  --vcf=variants_illumina.vcf \
  --output=phased_illumina.vcf
!java -jar fgbio-3.0.0.jar HapCutToVcf \
  --input=pacbio_phased.vcf \
  --vcf=variants_pacbio.vcf \
  --output=phased_pacbio.vcf

Nov 05, 2025 3:41:03 PM com.intel.gkl.NativeLibraryLoader load
INFO: Loading libgkl_compression.so from jar:file:/home/liam/Bioinformatics/fall25-csc-bioinf/week5/fgbio-3.0.0.jar!/com/intel/gkl/native/libgkl_compression.so
[2025/11/05 15:41:04 | FgBioMain | Info] Executing HapCutToVcf from fgbio version 3.0.0 as liam@WAMZERDON on JRE 17.0.16+8-Ubuntu-0ubuntu124.04.1 with snappy, IntelInflater, and IntelDeflater
[2025/11/05 15:41:05 | FgBioMain | Info] HapCutToVcf completed. Elapsed time: 0.02 minutes.
Nov 05, 2025 3:41:05 PM com.intel.gkl.NativeLibraryLoader load
INFO: Loading libgkl_compression.so from jar:file:/home/liam/Bioinformatics/fall25-csc-bioinf/week5/fgbio-3.0.0.jar!/com/intel/gkl/native/libgkl_compression.so
[2025/11/05 15:41:06 | FgBioMain | Info] Executing HapCutToVcf from fgbio version 3.0.0 as liam@WAMZERDON on JRE 17.0.16+8-Ubuntu-0ubuntu124.04.1 with snappy, IntelInflater, and IntelDeflater
[2025/11/05 15:41:06 | FgBioMain | Info] HapCutToVcf completed. Elapsed time: 

# Step 5
Analyzing variants not common between the vcf files

In [186]:
def parse_vcf(path):
    variants = set()
    with open(path) as f:
        for line in f:
            if line.startswith("#"):
                continue
            chrom, pos, _id, ref, alt, *_ = line.strip().split("\t")
            variants.add((chrom, int(pos), ref, alt))
    return variants

illumina = parse_vcf("phased_illumina.vcf")
pacbio   = parse_vcf("phased_pacbio.vcf")

shared = illumina & pacbio
illumina_only = illumina - pacbio
pacbio_only   = pacbio - illumina

print("Shared:", len(shared))
print("Illumina only:", len(illumina_only))
print("PacBio only:", len(pacbio_only))

Shared: 391
Illumina only: 5626
PacBio only: 243


Now take screenshots of the locations in IGV

In [194]:
!rm -rf snapshots
!mkdir snapshots

In [195]:
variant = next(iter(illumina_only))  # (chrom, pos, ref, alt)
chrom, pos, *_ = variant

print("Example Illumina-only variant:", variant)

with open("igv_batch.txt", "w") as f:
    f.write(f"""new
genome chr10.fa
load sample_illumina.bam
load sample_pacbio.bam
snapshotDirectory ./snapshots
goto {chrom}:{pos}
snapshot Illumina_only.png
exit
""")


!igv -b igv_batch.txt

variant = next(iter(pacbio_only))  # (chrom, pos, ref, alt)
chrom, pos, *_ = variant

print("Example PacBio-only variant:", variant)

with open("igv_batch.txt", "w") as f:
    f.write(f"""new
genome chr10.fa
load sample_illumina.bam
load sample_pacbio.bam
snapshotDirectory ./snapshots
goto {chrom}:{pos}
snapshot Pacbio_only.png
exit
""")
    
!igv -b igv_batch.txt

Example Illumina-only variant: ('chr10', 94409277, 'C', 'A')
INFO [Nov 05,2025 16:14] [Main] Startup  IGV Version user not_set
INFO [Nov 05,2025 16:14] [Main] Java 21.0.8 (build 21.0.8+9-Ubuntu-0ubuntu124.04.1) 2025-07-15
INFO [Nov 05,2025 16:14] [Main] Java Vendor: Ubuntu https://ubuntu.com/
INFO [Nov 05,2025 16:14] [Main] JVM: OpenJDK 64-Bit Server VM    
INFO [Nov 05,2025 16:14] [Main] OS: Linux 6.6.87.2-microsoft-standard-WSL2 amd64
INFO [Nov 05,2025 16:14] [Main] IGV Directory: /home/liam/igv
SEVERE [Nov 05,2025 16:14] [CommandListener] java.net.BindException: Address already in use
	at java.base/sun.nio.ch.Net.bind0(Native Method)
	at java.base/sun.nio.ch.Net.bind(Net.java:565)
	at java.base/sun.nio.ch.Net.bind(Net.java:554)
	at java.base/sun.nio.ch.NioSocketImpl.bind(NioSocketImpl.java:636)
	at java.base/java.net.ServerSocket.bind(ServerSocket.java:391)
	at java.base/java.net.ServerSocket.<init>(ServerSocket.java:278)
	at java.base/java.net.ServerSocket.<init>(ServerSocket.java:

![alt text](snapshots/Illumina_only.png)

![alt text](snapshots/Pacbio_only.png)

Based on these screenshots, these variants are most likely sequencing-related artifacts and not true variants since they occur only in one of the sequencing technologies. The vairants that appear in both sequencing technologies are more likely to be true variants.

# Step 6

This step is omitted